# S&P 500 sector universes from WRDS

This notebook retrieves the four sector-year universes used in the project:

- Energy 2008
- Energy 2024
- Communication Services 2024
- Materials 2024

A security is included if it was a constituent of the S&P 500 at any point during the relevant calendar year. CRSP provides S&P 500 membership and security names, while the CRSP/Compustat link table and Compustat company table provide GICS sector classifications.

 WRDS may prompt for credentials when that connection is created, unless local WRDS authentication has already been configured.


In [2]:
!pip install wrds
import pandas as pd
import wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 109.8 MB/s eta 0:00:00


In [3]:
# Open a WRDS connection for all queries in this notebook.
db = wrds.Connection()

Enter your WRDS username [root]:as3025
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
GICS_SECTOR_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate",
}


def get_sp500_sector_universe(db, year, sector):
    """Return S&P 500 securities in a GICS sector during a calendar year.

    A security is included when its CRSP S&P 500 membership interval overlaps
    the requested calendar year. CRSP name records and CRSP/Compustat links
    must also overlap the relevant membership interval.

    Parameters
    ----------
    db : wrds.Connection
        Existing WRDS connection.
    year : int
        Calendar year to retrieve.
    sector : str
        GICS sector name, for example ``"Energy"``.

    Returns
    -------
    pandas.DataFrame
        Security-level table containing ticker, company name, CRSP PERMNO,
        Compustat GVKEY, GICS sector code, and S&P 500 membership dates.
    """
    if sector not in GICS_SECTOR_MAP.values():
        raise ValueError(f"Unknown GICS sector: {sector}")

    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    analysis_end = pd.Timestamp(end_date)

    # S&P 500 securities whose membership overlaps the requested year.
    sp500 = db.raw_sql(
        """
        SELECT permno, start, ending
        FROM crsp.dsp500list
        WHERE start <= CAST(%(end_date)s AS date)
          AND (
                ending IS NULL
                OR ending >= CAST(%(start_date)s AS date)
              )
        ORDER BY permno, start
        """,
        params={"start_date": start_date, "end_date": end_date},
        date_cols=["start", "ending"],
    )

    # CRSP ticker and company-name histories overlapping the requested year.
    names = db.raw_sql(
        """
        SELECT permno, namedt, nameendt, ticker, comnam, ncusip
        FROM crsp.msenames
        WHERE namedt <= CAST(%(end_date)s AS date)
          AND (
                nameendt IS NULL
                OR nameendt >= CAST(%(start_date)s AS date)
              )
        """,
        params={"start_date": start_date, "end_date": end_date},
        date_cols=["namedt", "nameendt"],
    )

    sp500_names = sp500.merge(names, on="permno", how="left")

    membership_end = sp500_names["ending"].fillna(analysis_end)
    name_end = sp500_names["nameendt"].fillna(analysis_end)

    sp500_names = sp500_names[
        (sp500_names["namedt"] <= membership_end)
        & (name_end >= sp500_names["start"])
    ].copy()

    # Link CRSP securities to Compustat companies using valid primary links.
    ccm = db.raw_sql(
        """
        SELECT gvkey, lpermno AS permno, linkdt, linkenddt, linktype, linkprim
        FROM crsp.ccmxpf_lnkhist
        WHERE lpermno IS NOT NULL
          AND linktype IN ('LU', 'LC')
          AND linkprim IN ('P', 'C')
          AND linkdt <= CAST(%(end_date)s AS date)
          AND COALESCE(linkenddt, DATE '9999-12-31')
              >= CAST(%(start_date)s AS date)
        """,
        params={"start_date": start_date, "end_date": end_date},
        date_cols=["linkdt", "linkenddt"],
    )

    sp500_ccm = sp500_names.merge(ccm, on="permno", how="left")

    membership_end = sp500_ccm["ending"].fillna(analysis_end)
    link_end = sp500_ccm["linkenddt"].fillna(analysis_end)

    sp500_ccm = sp500_ccm[
        (sp500_ccm["linkdt"] <= membership_end)
        & (link_end >= sp500_ccm["start"])
    ].copy()

    # Compustat company table provides the GICS sector classification.
    gics = db.raw_sql(
        """
        SELECT gvkey, conm, gsector, ggroup, gind, gsubind
        FROM comp.company
        """
    )

    final = sp500_ccm.merge(gics, on="gvkey", how="left")

    final["gsector"] = final["gsector"].astype("string").str.zfill(2)
    final["gics_sector_name"] = final["gsector"].map(GICS_SECTOR_MAP)
    final["ticker"] = final["ticker"].astype("string").str.upper().str.strip()

    final = final.loc[final["gics_sector_name"] == sector].copy()

    # Multiple CRSP name-history rows can describe the same security within a
    # year. Keep one row per PERMNO/ticker for a clean security-level universe.
    final = (
        final[
            [
                "ticker",
                "comnam",
                "conm",
                "permno",
                "gvkey",
                "gsector",
                "gics_sector_name",
                "start",
                "ending",
            ]
        ]
        .dropna(subset=["ticker"])
        .drop_duplicates(subset=["permno", "ticker"])
        .sort_values(["ticker", "permno"])
        .reset_index(drop=True)
    )

    return final


def print_universe(label, universe):
    """Print the ticker and CRSP company name for a sector-year universe."""
    ticker_names = (
        universe[["ticker", "comnam"]]
        .drop_duplicates()
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    print(label)
    print(f"Number of tickers: {ticker_names['ticker'].nunique()}")
    print(ticker_names.to_string(index=False))
    print("\n" + "=" * 80 + "\n")

## Retrieve the four project universes

Each call below reuses the same `db` connection created above.

In [6]:
energy_2008 = get_sp500_sector_universe(db, 2008, "Energy")
energy_2024 = get_sp500_sector_universe(db, 2024, "Energy")
communication_services_2024 = get_sp500_sector_universe(
    db, 2024, "Communication Services"
)
materials_2024 = get_sp500_sector_universe(db, 2024, "Materials")

## Print ticker and company name for each universe

In [7]:
print_universe("Energy 2008", energy_2008)
print_universe("Energy 2024", energy_2024)
print_universe("Communication Services 2024", communication_services_2024)
print_universe("Materials 2024", materials_2024)

Energy 2008
Number of tickers: 41
ticker                       comnam
   APA                  APACHE CORP
   APC      ANADARKO PETROLEUM CORP
   BHI             BAKER HUGHES INC
   BJS              B J SERVICES CO
   BTU          PEABODY ENERGY CORP
   CAM   CAMERON INTERNATIONAL CORP
   CHK       CHESAPEAKE ENERGY CORP
   CNX            CONSOL ENERGY INC
   COG         CABOT OIL & GAS CORP
   COP               CONOCOPHILLIPS
   CVX             CHEVRON CORP NEW
   DVN        DEVON ENERGY CORP NEW
   EOG            EOG RESOURCES INC
    EP                 EL PASO CORP
   EQT      EQUITABLE RESOURCES INC
   ESV  E N S C O INTERNATIONAL INC
   HAL               HALLIBURTON CO
   HES                    HESS CORP
   MEE             MASSEY ENERGY CO
   MRO            MARATHON OIL CORP
   MUR              MURPHY OIL CORP
   NBL             NOBLE ENERGY INC
   NBR        NABORS INDUSTRIES LTD
    NE                   NOBLE CORP
   NOV   NATIONAL OILWELL VARCO INC
   OXY    OCCIDENTAL PETROLEUM

In [9]:
# Close the WRDS connection after all queries are complete.
db.close()